# 시간 설정

In [1]:
import time
import random
import os

os.environ['TZ'] = 'Asia/Seoul'
time.tzset()

random.seed(1490)

def print_time() -> None:
    print(time.strftime('%Y년 %m월 %d일 %A %p %I시 %M분 %S초'))
    return

def timestamp() -> str:
    return time.strftime('%Y-%m-%d-%H-%M-%S')

print_time()
print(timestamp())

2024년 08월 25일 Sunday PM 03시 38분 12초


# 제품 이상여부 판별 프로젝트


## 필수 라이브러리


In [3]:
import os
from pprint import pprint

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn import metrics
from sklearn.model_selection import train_test_split
from tqdm import tqdm

from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt  # 그래프 보기
import seaborn as sns  # 그래프 보기

print_time()

2024년 08월 25일 Sunday PM 03시 38분 14초


## 전역변수

In [4]:
global drops

# ROOT_PREFIX = "/content/drive/MyDrive/lgaimers5/" # 구글 드라이브용 루트
# ROOT_DIR = ROOT_PREFIX + "data" # 에이머스 제출 시 주석 처리 필요
ROOT_DIR = "data" # 구글 드라이브 실행 시 주석 처리 필요

RANDOM_STATE = 110 # 랜덤 시드
corr_thresh = 0.0001 # 상관관계 삭제 기준
empty_thresh = 30 # 결측값 삭제 기준 비율

# 랜덤 시드 설정
np.random.seed(RANDOM_STATE)

# 1. 필터 1번, 필터 2번 등과 같이 순서가 있는 값은 추가 처리 없이 넘버링,
# 그 외 기타 order들도 같이 넘버링
numbering = ['Equipment_Dam', 'Model.Suffix_Dam', 'Workorder_Dam', \
             'Model.Suffix_AutoClave', 'Workorder_AutoClave', 'Equipment_Fill1', \
             'Model.Suffix_Fill1', 'Workorder_Fill1', 'Equipment_Fill2', \
             'Model.Suffix_Fill2', 'Workorder_Fill2']

# 2. OK, NG와 같이 범주형인 경우 가능하면 원 핫 인코딩
onehot = ['HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Dam', \
          'GMES_ORIGIN_INSP_JUDGE_CODE Collect Result_AutoClave', \
          'GMES_ORIGIN_INSP_JUDGE_CODE Judge Value_AutoClave', \
          'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill1', \
          'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill2', \
          'Chamber Temp. Judge Value_AutoClave']

# 3. 숫자 값이 들어가야 하는데 OK가 혼입된 값임
custom = ['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam', \
          'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1', \
          'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2']

drops = [] # 값이 없거나 무의미해서 버릴 column들
dt_nan = []  # attributes to fill the missing value with any value
dt_del = []  # attributes to be deleted because there are too many missing values
need_general = []  # normalization list
del_list = [] # 상관관계 삭제 리스트
reduce_list = [] # 다중공선성 삭제 리스트
# 다중공선성: 서로 다른 속성끼리 깊은 연관을 갖는 것.
# 예를 들면 타겟은 따로 있는데 A 속성과 B 속성 둘 사이의 상관관계 값의 절댓값이 1에 가깝게 나오는 경우를 말함

print_time()

2024년 08월 25일 Sunday PM 03시 38분 14초


## 함수 정의

In [5]:
def print_col_uniqs(data):
    global drops
    """
    unique 개수가 1개 이하인 column 삭제
    """
    init_cols = len(data.columns)
    total_cnt = len(data)

    for col in data.columns:
        if col == 'target':
            continue
        uniqs = data[col].unique()
        nuq = data[col].nunique()
        vcnt = data[col].value_counts()
        if (nuq <= 1 and len(uniqs) <= 1) or (len(data) == data[col].isna().sum()):
            drops.append(col)
        else:
            if nuq <= 10:
                # print(f'{col} : {nuq}, {uniqs}')
                pass
            else:
                # print(f'{col} : {nuq}')
                pass
            # 결측값 비율 보기(null이 있는 속성만)
            cnt = data[col].isna().sum()

            if cnt == 0:
                continue

            num = float(cnt) / float(total_cnt)

            if num * 100.0 <= empty_thresh:
                dt_nan.append(col)
            else:
                dt_del.append(col)
    
    drops += dt_del
    print(f'drops: {len(drops)}\ndt_del: {len(dt_del)}')
    data.drop(drops, axis=1, inplace=True)
    data.drop_duplicates(keep='first', inplace=True, ignore_index=True) # 중복 제거
    data[list(data.columns)].dropna(axis=0, inplace=True) # 모든 속성이 결측값인 행 제거
    
    # data.fillna(-1, inplace = True)
    for col in data.columns:
        if str(data[col].dtypes) != 'object':
            data[col] = data[col].fillna(data[col].mean())
        else:
            data[col].fillna(-1, inplace = True)
    print(f'결측값 처리 후 결측값 개수: {data.isna().sum().sum()}')
    print(f'num of initial cols: {init_cols}')
    print(f'num of deleted cols: {len(drops)}')
    print(f'num of remained cols: {len(data.columns)}')
    return data, drops

def print_columns_to_list(data):
    """
    컬럼이 너무 많아 바로 출력이 안되니 직접 출력
    """
    cnt = {}
    for col in data.columns:
        dt = str(data[col].dtypes)
        nu = data[col].nunique()
        if dt in cnt.keys():
            cnt[dt] += 1
        else:
            cnt[dt] = 1
        print(f'* {col} : {dt}({nu})')
    print(f'\ninfo: {cnt}')

def object_unique(data):
    """
    dtype이 object로 분류된 컬럼의 unique 값 확인
    """
    idx = 1
    for col in data.columns:
        if str(data[col].dtypes) == 'object':
            print(f'{idx}. {col}\n\t{data[col].nunique()}', end='')
            if data[col].nunique() <= 20:
                print(f', {data[col].unique()}')
            else:
                print()
            idx += 1

def enc_numbering(data, cols):
    """
    단순 넘버링으로 라벨 인코딩
    """
    print('넘버링')
    for col in cols:
        if col not in data.columns:
            continue
        data[col] = data[col].astype(str)
        le = LabelEncoder()
        le.fit(sorted(list(data[col].unique())))
        data[col] = le.transform(data[col])
    return data

def enc_onehot(data, cols):
    """
    원 핫 인코딩
    """
    print('원 핫 인코딩')
    for c in cols:
        if c not in data.columns:
            cols.remove(c)
    data = pd.get_dummies(data, columns = cols)
    return data

def enc_only_obj(data, cols):
    """
    다른 컬럼의 OK가 혼입된 값 정리, 소수 값을 리터럴 소수로 변환 후 dtype도 소수로 지정
    """
    print('문자열 숫자를 숫자로')
    for col in cols:
        if col not in data.columns:
            continue
        idx = 1
        uniqs = data[col].unique()
        for u in uniqs:
            try:
                data.loc[data[col] == u, col] = float(u)
            except:
                if u == 'OK' or u == -1:
                    data.loc[data[col] == u, col] = -1
                else:
                    data.loc[data[col] == u, col] = idx
                    idx += 1
        data[col] = data[col].astype(float)
        # print(data[col])
    return data

def obj_encoding(data, mode = 'train'):
    """
    모든 라벨 인코딩 실행
    """
    # 기본 넘버링
    data = enc_numbering(data, numbering)
    # 원 핫 인코딩
    # data = enc_onehot(data, onehot)
    data = enc_numbering(data, onehot)
    # OK가 혼입된 값 정리
    data = enc_only_obj(data, custom)
    # target 인코딩
    # 있는 경우에만
    if mode == 'train':
        print('target 값을 숫자로')
        data.loc[data['target'] == 'AbNormal', 'target'] = 1
        data.loc[data['target'] == 'Normal', 'target'] = 0
    return data

def show_heatmap(data, cnt = 12):
    """
    컬럼을 지정된 수만큼 묶어서 히트맵 출력
    * 색깔 참고: https://colorbrewer2.org/
    """
    cols = list(data.columns)
    corr = data.corr()
    print(corr.info())

    # plt.figure(figsize=(15, 10))
    # sns.heatmap(data.corr(), annot=True, cmap="RdBu", vmax=1)
    # plt.show()

def del_by_corr(data, thresh = 0.5, del_list = []):
    """
    지정된 값을 기준으로 상관관계가 낮은 컬럼 삭제
    """
    corr = data.corr()
    target_corr = corr['target']
    target_corr.drop('target', inplace=True)
    del_list = list(target_corr[target_corr.isnull() == True].index)
    temp = target_corr[-thresh < target_corr]
    del_list += list(temp[temp < thresh].index)

    data.drop(del_list, axis=1, inplace=True)
    print(f'num of cols deleted by correlation: {len(del_list)}')
    print(f'remain cols: {len(data.columns)}')

    return data, del_list

def corr_reduce(data, thresh=0.99, reduce_list=reduce_list):
    # 상관계수 행렬 계산
    correlation_matrix = data.corr().abs()

    # 상관관계 계수가 threshold 이상인 피처들 중 하나를 제거하기 위한 마스크 생성
    upper_triangle = correlation_matrix.where(
        np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
    )

    # target_column과 연관된 피처를 제외하기 위해, target_column과 관련된 피처를 식별
    related_to_target = correlation_matrix['target'].index

    # 제거할 피처 리스트 생성
    to_drop = [column for column in upper_triangle.columns
               if column != 'target' and any(upper_triangle[column] > thresh)]

    # 상관관계가 높은 피처 제거
    df_reduced = data.drop(columns=to_drop)

    print("제거된 속성 수:", len(to_drop))

    return df_reduced, to_drop

def pre_processing(data, del_list=del_list, reduce_list=reduce_list):
    """
    데이터 전처리 통합 실행
    """
    # 데이터 뒤섞기
    print(f'# 데이터 뒤섞기')
    data = data.sample(frac=1)  # randomize data

    # unique 개수가 1개 이하인 column 삭제
    print('# unique 개수가 1개 이하인 column 삭제')
    data, drops = print_col_uniqs(data)

    # 모든 라벨 인코딩 실행
    print('# 모든 라벨 인코딩 실행')
    data = obj_encoding(data, 'train')

    # 상관관계가 낮은 컬럼 삭제
    print('# 상관관계가 낮은 컬럼 삭제')
    data, del_list = del_by_corr(data, thresh=corr_thresh, del_list=del_list)
    
    # 다중공선성이 높은 컬럼 삭제
    print('# 다중공선성이 높은 컬럼 삭제')
    data, reduce_list = corr_reduce(data)

    print('target 값 정수화')
    data['target'] = data['target'].astype(int)

    # 전처리 완료
    print('# 전처리 완료')
    print_time()

    return data, drops, del_list, reduce_list

def pre_processing_test(data):
    """
    테스트 데이터(레이블 X) 전처리 통합 실행
    """
    # unique 개수가 1개 이하인 column 삭제
    print('# unique 개수가 1개 이하인 column 삭제')
    print('속성 삭제')
    data.drop(drops, axis=1, inplace=True)
    for col in data.columns:
        if str(data[col].dtypes) != 'object':
            data[col] = data[col].fillna(data[col].mean())
        else:
            data[col].fillna(-1, inplace = True)
    print(f'결측값 처리 후 결측값 개수: {data.isna().sum().sum() - data["target"].isna().sum()}')

    # 모든 라벨 인코딩 실행
    print('# 모든 라벨 인코딩 실행')
    data = obj_encoding(data, 'test')

    # 상관관계가 낮은 컬럼 삭제
    print('# 상관관계가 낮은 컬럼 삭제')
    print(f'num of cols deleted by correlation: {len(del_list)}')
    data.drop(del_list, axis=1, inplace=True)
    print(f'remain cols: {len(data.columns)}')
    
    # 다중공선성이 높은 컬럼 삭제
    print('# 다중공선성이 높은 컬럼 삭제')
    data = data.drop(columns=reduce_list)

    # 테스트 데이터 전처리 완료
    print('# 테스트 데이터 전처리 완료')
    print_time()

    return data

print_time()

2024년 08월 25일 Sunday PM 03시 38분 14초


## 1. 데이터 불러오기


### 데이터 읽어오기


In [6]:
print_time()

# Load data
train_data = pd.read_csv(os.path.join(ROOT_DIR, "train.csv"))

# 데이터 전처리
train_data, drops, del_list, reduce_list = pre_processing(train_data)

2024년 08월 25일 Sunday PM 03시 38분 14초
# 데이터 뒤섞기
# unique 개수가 1개 이하인 column 삭제
drops: 321
dt_del: 8
결측값 처리 후 결측값 개수: 0
num of initial cols: 464
num of deleted cols: 321
num of remained cols: 143
# 모든 라벨 인코딩 실행
넘버링
넘버링
문자열 숫자를 숫자로
target 값을 숫자로
# 상관관계가 낮은 컬럼 삭제
num of cols deleted by correlation: 0
remain cols: 143
# 다중공선성이 높은 컬럼 삭제
제거된 속성 수: 81
target 값 정수화
# 전처리 완료
2024년 08월 25일 Sunday PM 03시 38분 20초


In [7]:
print(f'drops: {len(drops)}, del_list: {len(del_list)}, reduce_list: {len(reduce_list)}')

drops: 321, del_list: 0, reduce_list: 81


### columns info

In [8]:
# 전체 컬럼 정보 출력
print_columns_to_list(train_data)

* Equipment_Dam : int64(2)
* Model.Suffix_Dam : int64(7)
* Workorder_Dam : int64(663)
* CURE SPEED Collect Result_Dam : int64(5)
* DISCHARGED SPEED OF RESIN Collect Result_Dam : int64(3)
* DISCHARGED TIME OF RESIN(Stage1) Collect Result_Dam : float64(19)
* DISCHARGED TIME OF RESIN(Stage2) Collect Result_Dam : float64(29)
* Dispense Volume(Stage1) Collect Result_Dam : float64(23)
* Dispense Volume(Stage2) Collect Result_Dam : float64(33)
* HEAD NORMAL COORDINATE X AXIS(Stage2) Collect Result_Dam : float64(41)
* HEAD NORMAL COORDINATE X AXIS(Stage3) Collect Result_Dam : float64(40)
* HEAD NORMAL COORDINATE Y AXIS(Stage1) Collect Result_Dam : float64(36)
* HEAD NORMAL COORDINATE Z AXIS(Stage1) Collect Result_Dam : float64(16)
* HEAD NORMAL COORDINATE Z AXIS(Stage2) Collect Result_Dam : float64(24)
* HEAD Standby Position X Collect Result_Dam : float64(23)
* HEAD Standby Position Y Collect Result_Dam : int64(2)
* PalletID Collect Result_Dam : float64(374)
* Production Qty Collect Result_Da

### object uniques

In [9]:
# object 데이터의 unique 값 확인
object_unique(train_data)

print_time()

2024년 08월 25일 Sunday PM 03시 38분 20초


In [10]:
# info 확인
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40506 entries, 0 to 40505
Data columns (total 62 columns):
 #   Column                                                      Non-Null Count  Dtype  
---  ------                                                      --------------  -----  
 0   Equipment_Dam                                               40506 non-null  int64  
 1   Model.Suffix_Dam                                            40506 non-null  int64  
 2   Workorder_Dam                                               40506 non-null  int64  
 3   CURE SPEED Collect Result_Dam                               40506 non-null  int64  
 4   DISCHARGED SPEED OF RESIN Collect Result_Dam                40506 non-null  int64  
 5   DISCHARGED TIME OF RESIN(Stage1) Collect Result_Dam         40506 non-null  float64
 6   DISCHARGED TIME OF RESIN(Stage2) Collect Result_Dam         40506 non-null  float64
 7   Dispense Volume(Stage1) Collect Result_Dam                  40506 non-null  float64
 

In [11]:
train_data.isna().sum().sum()

0

In [12]:
# 컬럼 리스트 확인
# 리스트로 변환하면 생략하지 않고 출력됨
# list(train_data.columns)

### ADASYN 오버 샘플링

https://dapin1490.github.io/satinbower/posts/it-deeplearning-data-2/#3-adasyn-%EC%98%A4%EB%B2%84-%EC%83%98%ED%94%8C%EB%A7%81

In [13]:
!pip install imblearn

Defaulting to user installation because normal site-packages is not writeable


In [14]:
import imblearn  # 샘플링 관련 라이브러리
print("\nimblearn 버전 확인\n{}".format(imblearn.__version__))

# ADASYN 오버 샘플링
# 참고 https://dining-developer.tistory.com/27
from imblearn.over_sampling import ADASYN

temp_x = train_data.drop('target', axis=1)
temp_y = train_data['target']

temp_x, temp_y = ADASYN(sampling_strategy=0.3).fit_resample(temp_x, temp_y)

df_concat = pd.concat([temp_x, temp_y], axis=1).reset_index(drop=True)
df_concat.value_counts('target')


imblearn 버전 확인
0.12.3


target
0    38156
1    11029
Name: count, dtype: int64

### 언더 샘플링


데이타 불균형을 해결하기 위해 언더 샘플링을 진행합니다.


In [15]:
normal_ratio = 3.2  # 1.0 means 1:1 ratio

df_normal = df_concat[df_concat["target"] == 0]
df_abnormal = df_concat[df_concat["target"] == 1]

num_normal = len(df_normal)
num_abnormal = len(df_abnormal)
print(f"Total: Normal: {num_normal}, AbNormal: {num_abnormal}")

df_normal = df_normal.sample(n=int(num_abnormal * normal_ratio), replace=False, random_state=RANDOM_STATE)
df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
print(df_concat.value_counts("target"))

print()
print_time()

Total: Normal: 38156, AbNormal: 11029
target
0    35292
1    11029
Name: count, dtype: int64

2024년 08월 25일 Sunday PM 03시 38분 23초


### 데이터 분할


In [16]:
print_time()
print()

df_train, df_val = train_test_split(
    df_concat,
    test_size=0.2,
    stratify=df_concat["target"],
    random_state=RANDOM_STATE,
)

def print_stats(df: pd.DataFrame):
    num_normal = len(df[df["target"] == 0])
    num_abnormal = len(df[df["target"] == 1])

    print(f"  Total: Normal: {num_normal}, AbNormal: {num_abnormal}" + f" ratio: {num_abnormal / num_normal}")

# Print statistics
print(f"  \tNormal    \tAbNormal")
print_stats(df_train)
print_stats(df_val)

2024년 08월 25일 Sunday PM 03시 38분 23초

  	Normal    	AbNormal
  Total: Normal: 28233, AbNormal: 8823 ratio: 0.31250664116459465
  Total: Normal: 7059, AbNormal: 2206 ratio: 0.312508853945318


In [17]:
features = []

for col in df_train.columns:
    try:
        # df_train[col] =
        df_train[col].astype(int) # float를 그대로 두기 위해 일부러 줄을 내렸으나, object 컬럼을 제외한다는 것은 유지하기 위해 이 부분은 놔둠
        features.append(col)
    except:
        continue

if 'target' in features:
    features.remove('target')

train_x = df_train[features]
train_y = df_train["target"]

print_time()
train_x.info()

2024년 08월 25일 Sunday PM 03시 38분 23초
<class 'pandas.core.frame.DataFrame'>
Index: 37056 entries, 11561 to 32298
Data columns (total 61 columns):
 #   Column                                                      Non-Null Count  Dtype  
---  ------                                                      --------------  -----  
 0   Equipment_Dam                                               37056 non-null  int64  
 1   Model.Suffix_Dam                                            37056 non-null  int64  
 2   Workorder_Dam                                               37056 non-null  int64  
 3   CURE SPEED Collect Result_Dam                               37056 non-null  int64  
 4   DISCHARGED SPEED OF RESIN Collect Result_Dam                37056 non-null  int64  
 5   DISCHARGED TIME OF RESIN(Stage1) Collect Result_Dam         37056 non-null  float64
 6   DISCHARGED TIME OF RESIN(Stage2) Collect Result_Dam         37056 non-null  float64
 7   Dispense Volume(Stage1) Collect Result_Dam        

In [18]:
# 학습 데이터 기준으로 f1 score 채점
test_x = df_val[features]
test_y = df_val["target"]
pred_y = []

print_time()

2024년 08월 25일 Sunday PM 03시 38분 23초


테스트 데이터 불러오기

In [19]:
test_data = pd.read_csv(os.path.join(ROOT_DIR, "test.csv"))
test_data.drop('Set ID', axis=1, inplace=True)
test_data = pre_processing_test(test_data)

print_time()
test_data.info()

# unique 개수가 1개 이하인 column 삭제
속성 삭제
결측값 처리 후 결측값 개수: 0
# 모든 라벨 인코딩 실행
넘버링
넘버링
문자열 숫자를 숫자로
# 상관관계가 낮은 컬럼 삭제
num of cols deleted by correlation: 0
remain cols: 143
# 다중공선성이 높은 컬럼 삭제
# 테스트 데이터 전처리 완료
2024년 08월 25일 Sunday PM 03시 38분 23초
2024년 08월 25일 Sunday PM 03시 38분 23초
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17361 entries, 0 to 17360
Data columns (total 62 columns):
 #   Column                                                      Non-Null Count  Dtype  
---  ------                                                      --------------  -----  
 0   Equipment_Dam                                               17361 non-null  int64  
 1   Model.Suffix_Dam                                            17361 non-null  int64  
 2   Workorder_Dam                                               17361 non-null  int64  
 3   CURE SPEED Collect Result_Dam                               17361 non-null  int64  
 4   DISCHARGED SPEED OF RESIN Collect Result_Dam                17361 non-null  int64  

### 사이킷런 GenericUnivariateSelect로 속성 선택

In [20]:
from sklearn.feature_selection import GenericUnivariateSelect, f_classif
import pandas as pd

# 데이터 로드 및 데이터프레임 생성
df, y = train_x, train_y

# GenericUnivariateSelect를 사용하여 피처 선택
selector = GenericUnivariateSelect(score_func=f_classif, mode='fdr')
X_new = selector.fit_transform(df, y)

# 선택된 피처의 인덱스 얻기
selected_columns = selector.get_support(indices=True)

# 선택된 컬럼명을 원본 데이터프레임에서 가져오기
selected_feature_names = df.columns[selected_columns]

# 선택된 피처로 새로운 데이터프레임 생성
df_selected = pd.DataFrame(X_new, columns=selected_feature_names)

print(df_selected.info())

In [21]:
features = list(df_selected.columns)

train_x = df_train[features]
test_x = df_val[features]
test_data = test_data[features]

test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17361 entries, 0 to 17360
Data columns (total 61 columns):
 #   Column                                                      Non-Null Count  Dtype  
---  ------                                                      --------------  -----  
 0   Equipment_Dam                                               17361 non-null  int64  
 1   Model.Suffix_Dam                                            17361 non-null  int64  
 2   Workorder_Dam                                               17361 non-null  int64  
 3   CURE SPEED Collect Result_Dam                               17361 non-null  int64  
 4   DISCHARGED SPEED OF RESIN Collect Result_Dam                17361 non-null  int64  
 5   DISCHARGED TIME OF RESIN(Stage1) Collect Result_Dam         17361 non-null  float64
 6   DISCHARGED TIME OF RESIN(Stage2) Collect Result_Dam         17361 non-null  float64
 7   Dispense Volume(Stage1) Collect Result_Dam                  17361 non-null  float64
 

## 3. 모델 학습


### 모델 정의


In [22]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.ensemble import VotingClassifier

models = {}
models_name = {}
knns = [
    KNeighborsClassifier(n_neighbors=12, weights='distance', p=1),
    KNeighborsClassifier(n_neighbors=20, weights='distance', p=1),
    KNeighborsClassifier(n_neighbors=2, weights='distance', p=1),
    KNeighborsClassifier(n_neighbors=8, weights='distance', p=1),
    KNeighborsClassifier(n_neighbors=16, weights='distance', p=1)
]

# K최근접이웃
models['knn'] = KNeighborsClassifier(n_neighbors=12, weights='distance', p=1)
models_name['knn'] = 'K-Neighbors Classifier'

# 랜덤포레스트
models['rf'] = RandomForestClassifier(n_estimators=20, random_state=RANDOM_STATE, verbose=1, class_weight='balanced')
models_name['rf'] = 'Random Forest Classifier'

# AdaBoost Classifier
models['ada'] = AdaBoostClassifier()
models_name['ada'] = 'Ada Boost Classifier'

# Bagging Classifier
models['bag'] = BaggingClassifier()
models_name['bag'] = 'Bagging Classifier'

# Voting Classifier
models['vc'] = VotingClassifier(
    estimators=[
        ('KNN1', knns[0]),
        ('KNN2', knns[1]),
        ('KNN3', knns[2]),
        ('KNN4', knns[3]),
        ('KNN5', knns[4]),
    ],
    voting='soft',
    weights={0: 1, 1: 3}
)
models_name['vc'] = 'Voting Classifier'

for name in models.keys():
    print(f'* \'{models_name[name]}\': {models[name]}')

{'knn': KNeighborsClassifier(n_neighbors=12, p=1, weights='distance'),
 'dt': DecisionTreeClassifier(class_weight={0: 1, 1: 2}, random_state=110,
                        splitter='random'),
 'rf': RandomForestClassifier(class_weight='balanced', n_estimators=20,
                        random_state=110, verbose=1),
 'rc': RidgeClassifier(),
 'gbc': GradientBoostingClassifier(),
 'ada': AdaBoostClassifier(),
 'bag': BaggingClassifier(),
 'etc': ExtraTreesClassifier(),
 'qda': QuadraticDiscriminantAnalysis(),
 'nc': NearestCentroid(),
 'vc': VotingClassifier(estimators=[('KNN1',
                               KNeighborsClassifier(n_neighbors=12, p=1,
                                                    weights='distance')),
                              ('KNN2',
                               KNeighborsClassifier(n_neighbors=20, p=1,
                                                    weights='distance')),
                              ('KNN3',
                               KNeighborsClas

### 모델 학습


In [ ]:
total_start_time = time.time()

print_time()

In [23]:
test_x.shape

(9265, 61)

In [24]:
test_y

33268    0
44721    1
42046    1
16825    0
28571    0
        ..
8345     0
35615    1
22667    0
17311    0
29665    0
Name: target, Length: 9265, dtype: int64

#### 랜덤 서치 모델 학습

In [25]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# Define the hyperparameter grid for each model
param_grids = {
    'knn': {
        'n_neighbors': randint(10, 20),
        'weights': ['uniform', 'distance'],
        'p': randint(1, 2)
    },
    'rf': {
        'n_estimators': randint(10, 200),
        'max_depth': randint(3, 20),
        'min_samples_split': randint(2, 10),
        'min_samples_leaf': randint(1, 10),
        'bootstrap': [True, False],
        'random_state': [RANDOM_STATE]
    },
    # Add more grids for other models if necessary
}

# Perform RandomizedSearchCV for each model
for name in param_grids.keys():
    print(f'{models_name[name]} random search and training...\n')
    start = time.time()

    # Perform RandomizedSearchCV
    search = RandomizedSearchCV(models[name], param_grids[name], n_iter=20, scoring='f1', n_jobs=1, cv=5, random_state=RANDOM_STATE, verbose=1)
    search.fit(train_x, train_y)

    models[name] = search.best_estimator_
    print(f'Best hyperparameters for {models_name[name]}\n{search.best_params_}')

    end = time.time()
    print(f'{models_name[name]} trained with random search in {int(end - start) //  60} min {int(end - start) % 60} second\n')

knn random search and training...

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best hyperparameters for knn
{'n_neighbors': 10, 'p': 1, 'weights': 'distance'}
knn trained with random search in 8 min 7 second

dt random search and training...

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best hyperparameters for dt
{'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 3, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'random_state': 110, 'splitter': 'best'}
dt trained with random search in 0 min 3 second

rf random search and training...

Fitting 5 folds for each of 20 candidates, totalling 100 fits


[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.7s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.7s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.7s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.7s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.7s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    1.1s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    1.1s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    1.1s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Do

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    1.3s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    1.3s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    2.3s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    2.3s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    2.3s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    2.3s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    2.3s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    2.1s
[Parallel(n_jobs=1)]: Do

Best hyperparameters for rf
{'bootstrap': False, 'max_depth': 19, 'min_samples_leaf': 4, 'min_samples_split': 8, 'n_estimators': 127, 'random_state': 110}
rf trained with random search in 5 min 45 second

etc random search and training...

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best hyperparameters for etc
{'random_state': 110, 'n_estimators': 206, 'max_features': 'sqrt', 'criterion': 'gini', 'class_weight': {0: 3, 1: 1}}
etc trained with random search in 17 min 25 second

nc random search and training...

Fitting 5 folds for each of 2 candidates, totalling 10 fits


/home/elicer/.local/lib/python3.10/site-packages/sklearn/model_selection/_search.py:305: UserWarning: The total space of parameters 2 is smaller than n_iter=20. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best hyperparameters for nc
{'metric': 'euclidean'}
nc trained with random search in 0 min 0 second



#### 랜덤 서치 안 하는 모델 학습

안 하는 사유
1. 너무 오래걸림
2. 딱히 만질 하이퍼파라미터가 안보임(몰라서)

In [26]:
models.keys()

dict_keys(['knn', 'dt', 'rf', 'rc', 'gbc', 'ada', 'bag', 'etc', 'qda', 'nc', 'vc'])

In [27]:
# 'ada': {
#     'estimator': [models['rf'], models['vc']],
#     'n_estimators': randint(50, 100).rvs(size=1),
#     'random_state': [RANDOM_STATE]
# },
# 'bag': {
#     'estimator': [models['knn'], models['rf'], models['ada'], models['vc']],
#     'n_estimators': randint(10, 50).rvs(size=1),
#     'random_state': [RANDOM_STATE]
# },

# AdaBoost Classifier
# knn이 sample_weight를 지원하지 않아 사용 불가
models['ada'] = AdaBoostClassifier(
    estimator=models['vc'],
    n_estimators=50,
    random_state=RANDOM_STATE
)

# Bagging Classifier
models['bag'] = BaggingClassifier(
    estimator=models['vc'],
    n_estimators=10,
    random_state=RANDOM_STATE
)

to_fit = ['vc', 'ada', 'bag']

for name in to_fit:
    print(f'{models_name[name]} training...')
    start = time.time()

    models[name].fit(train_x, train_y)

    end = time.time()
    print(f'{models_name[name]} trained with random search in {int(end - start) //  60} min {int(end - start) % 60} second\n')

vc random search and training...
vc trained with random search in 0 min 0 second

rc random search and training...
rc trained with random search in 0 min 0 second

gbc random search and training...
gbc trained with random search in 0 min 3 second

ada random search and training...
ada trained with random search in 0 min 1 second

bag random search and training...
bag trained with random search in 0 min 0 second

qda random search and training...
qda trained with random search in 0 min 0 second



/home/elicer/.local/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:926: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")


In [ ]:
total_train_time = int(time.time() - total_start_time)

print(f'total train time: {total_train_time // 60} min {total_train_time % 60} second')

### 기록 확인

In [28]:
# for name in best_models.keys():
#     models[name] = best_models[name]

for name in models.keys():
    print(f'* \'{models_name[name]}\': {models[name]}')

{'knn': KNeighborsClassifier(n_neighbors=10, p=1, weights='distance'),
 'dt': DecisionTreeClassifier(class_weight='balanced', criterion='entropy',
                        max_depth=3, max_features='log2', random_state=110),
 'rf': RandomForestClassifier(bootstrap=False, class_weight='balanced', max_depth=19,
                        min_samples_leaf=4, min_samples_split=8,
                        n_estimators=127, random_state=110, verbose=1),
 'rc': RidgeClassifier(class_weight={0: 1, 1: 2}, max_iter=10000, random_state=110),
 'gbc': GradientBoostingClassifier(max_depth=1, random_state=110),
 'ada': AdaBoostClassifier(estimator=DecisionTreeClassifier(class_weight='balanced',
                                                     criterion='entropy',
                                                     max_depth=3,
                                                     max_features='log2',
                                                     random_state=110),
                    random_sta

In [29]:
preds = {}

for name in models.keys():
    preds[name] = models[name].predict(test_x)  # Use best_models instead of models
    
    prec = precision_score(test_y, preds[name], average=None)
    recall = recall_score(test_y, preds[name], average=None)
    f1score = f1_score(test_y, preds[name], average=None)
    
    print(f"{models_name[name]} 학습 결과\n***************************")
    print(f"- 정확도: {accuracy_score(test_y, preds[name]) :.4f}")
    print(f"- precision: {prec[1] :.4f}")
    print(f"- recall: {recall[1] :.4f}")
    print(f"- f1 score: {f1score[1] :.4f}")
    print(f'precision과 recall의 차이: {prec[1] - recall[1] :.4f}')
    print()

knn 학습 결과
***************************
- 정확도: 0.8710
- precision: 0.7311
- recall: 0.7248
- f1 score: 0.7280
precision과 recall의 차이: 0.0063

dt 학습 결과
***************************
- 정확도: 0.5958
- precision: 0.3261
- recall: 0.6541
- f1 score: 0.4352
precision과 recall의 차이: -0.3280



[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.1s


rf 학습 결과
***************************
- 정확도: 0.9427
- precision: 0.9544
- recall: 0.7974
- f1 score: 0.8689
precision과 recall의 차이: 0.1571

rc 학습 결과
***************************
- 정확도: 0.7358
- precision: 0.4166
- recall: 0.2738
- f1 score: 0.3304
precision과 recall의 차이: 0.1428

gbc 학습 결과
***************************
- 정확도: 0.7704
- precision: 1.0000
- recall: 0.0358
- f1 score: 0.0691
precision과 recall의 차이: 0.9642

ada 학습 결과
***************************
- 정확도: 0.8897
- precision: 0.7766
- recall: 0.7534
- f1 score: 0.7648
precision과 recall의 차이: 0.0232

bag 학습 결과
***************************
- 정확도: 0.8677
- precision: 0.7300
- recall: 0.7049
- f1 score: 0.7173
precision과 recall의 차이: 0.0252

etc 학습 결과
***************************
- 정확도: 0.9434
- precision: 0.9662
- recall: 0.7901
- f1 score: 0.8693
precision과 recall의 차이: 0.1761

qda 학습 결과
***************************
- 정확도: 0.7701
- precision: 0.9222
- recall: 0.0376
- f1 score: 0.0723
precision과 recall의 차이: 0.8846

nc 학습 결과
********************

In [ ]:
for name in models.keys():
    confu_mat = confusion_matrix(test_y, preds[name], normalize='true')
    
    print(f"{models_name[name]} 학습 결과\n===========================")
    print(f"분류 결과:\n{classification_report(test_y, preds[name], target_names=['Normal', 'AbNormal'])}")
    print(f"혼동 행렬:\n{confu_mat}")
    sns.heatmap(confu_mat, annot=True, cmap='Blues', vmax = 1, vmin = 0)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'{models_name[name]} confusion mat.')
    plt.show()
    print()

## 4. 제출하기


### 테스트 데이터 예측


In [31]:
test_data.isna().sum().sum()

0

In [32]:
df_test_x = test_data

# 학습 데이터와 동일하게 정수화 안 함
# for col in df_test_x.columns:
#     try:
#         df_test_x.loc[:, col] = df_test_x[col].astype(int)
#     except:
#         continue

print_time()
df_test_x.info()

2024년 08월 25일 Sunday PM 04시 11분 56초
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17361 entries, 0 to 17360
Data columns (total 61 columns):
 #   Column                                                      Non-Null Count  Dtype  
---  ------                                                      --------------  -----  
 0   Equipment_Dam                                               17361 non-null  int64  
 1   Model.Suffix_Dam                                            17361 non-null  int64  
 2   Workorder_Dam                                               17361 non-null  int64  
 3   CURE SPEED Collect Result_Dam                               17361 non-null  int64  
 4   DISCHARGED SPEED OF RESIN Collect Result_Dam                17361 non-null  int64  
 5   DISCHARGED TIME OF RESIN(Stage1) Collect Result_Dam         17361 non-null  float64
 6   DISCHARGED TIME OF RESIN(Stage2) Collect Result_Dam         17361 non-null  float64
 7   Dispense Volume(Stage1) Collect Result_Dam       

In [33]:
models.keys()

dict_keys(['knn', 'dt', 'rf', 'rc', 'gbc', 'ada', 'bag', 'etc', 'qda', 'nc', 'vc'])

In [ ]:
models_name.keys()

In [34]:
test_preds = {}

for name in models.keys():
    test_preds[name] = models[name].predict(df_test_x)
    print(f'{models_name[name]} predicted {round(sum(test_preds[name]) / len(test_preds[name]) * 100, 2)}% AbNormal')
#     print()

knn predicted 9.39% AbNormal

dt predicted 42.53% AbNormal



[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.1s


rf predicted 1.71% AbNormal

rc predicted 12.46% AbNormal

gbc predicted 0.0% AbNormal

ada predicted 7.18% AbNormal

bag predicted 9.31% AbNormal

etc predicted 1.28% AbNormal

qda predicted 0.28% AbNormal

nc predicted 37.87% AbNormal

vc predicted 7.99% AbNormal



In [35]:
print_time()

# 제출용 모델 이름만 골라서 넣으면 됨
test_pred = test_preds['vc']

print(f'{round(sum(test_pred) / len(test_pred) * 100, 2)}% AbNormal')

test_pred

2024년 08월 25일 Sunday PM 04시 15분 03초
7.99% AbNormal


array([0, 0, 0, ..., 1, 0, 0])

### 제출 파일 작성


In [36]:
# 제출 데이터 읽어오기 (df_test는 전처리된 데이터가 저장됨)
# df_sub = pd.read_csv(ROOT_PREFIX + "submission.csv") # 에이머스 제출 시 주석 처리 필요
df_sub = pd.read_csv("submission.csv") # 구글 드라이브 실행 시 주석 처리 필요
df_sub["target"] = test_pred
df_sub.loc[df_sub['target'] >= 0.5, 'target'] = 1
df_sub.loc[df_sub['target'] < 0.5, 'target'] = 0

print(f"{sum(df_sub['target']) / len(df_sub['target']) * 100 :.2f}% AbNormal")

df_sub['target'] = df_sub['target'].astype(str)
df_sub.loc[df_sub['target'] == '1', 'target'] = 'AbNormal'
df_sub.loc[df_sub['target'] == '0', 'target'] = 'Normal'

print_time()

7.99% AbNormal
2024년 08월 25일 Sunday PM 04시 15분 03초


In [37]:
print_time()

df_sub.head(10)

2024년 08월 25일 Sunday PM 04시 15분 03초


,Set ID,target
0,0001be084fbc4aaa9d921f39e595961b,Normal
1,0005bbd180064abd99e63f9ed3e1ac80,Normal
2,000948934c4140d883d670adcb609584,Normal
3,000a6bfd02874c6296dc7b2e9c5678a7,Normal
4,0018e78ce91343678716e2ea27a51c95,Normal
5,001fda4596f545d0a3b0ce85fbea77d2,Normal
6,0020734a7b29472298358ad58645a0c9,Normal
7,00234c5914cd4c4a888d13f8b3773135,Normal
8,00297b6c93e44d49ac534758a23dc74e,Normal
9,002d904240d84b188d410d16383a9c3a,Normal


In [38]:
# 제출 파일 저장
# df_sub.to_csv(ROOT_PREFIX + "submission.csv", index=False) # 에이머스 제출 시 주석 처리 필요
df_sub.to_csv("submission.csv", index=False) # 구글 코랩 실행 시 주석 처리 필요

print_time()

2024년 08월 25일 Sunday PM 04시 15분 03초


**우측 상단의 제출 버튼을 클릭해 결과를 확인하세요**
